# **DATA 01 — Snapshot de la chaîne d'options AAPL (yfinance)**

Ce notebook télécharge la chaîne d'options AAPL via yfinance et l'écrit en CSV dans
`data/`. C'est le **seul** endroit du projet où yfinance est utilisé : le notebook
EQUITY_03 lit le CSV et ne touche jamais au réseau, pour que ses résultats (et les
commentaires écrits dessus) restent stables d'un run à l'autre.

À n'exécuter que pour rafraîchir le snapshot. Dans ce cas, mettre à jour le nom du
fichier lu au début d'EQUITY_03 et relire ses commentaires : ils décrivent les données
du jour du snapshot.

In [1]:
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import yfinance as yf

TICKER = "AAPL"

tk = yf.Ticker(TICKER)

# Le spot et la date du snapshot viennent du dernier close (plus fiable que
# fast_info hors séance).
history = tk.history(period="1d")
quote_date = history.index[-1].date()
spot = float(history["Close"].iloc[-1])
print(f"{TICKER} : spot {spot:.2f} au {quote_date}")

/Users/jules.remlinger/Desktop/Projets Code/Pricer/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AAPL : spot 305.70 au 2026-07-02


In [2]:
# Sélection des expirations : on écarte les tout premiers jours (gamma pur, quotes
# désordonnées) et au-delà de ~18 mois (open interest famélique). Après 3 mois, on ne
# garde que les échéances mensuelles (3e vendredi), les weeklies y sont illiquides.
def is_third_friday(d: date) -> bool:
    return d.weekday() == 4 and 15 <= d.day <= 21

expiries = []
for exp_str in tk.options:
    exp = date.fromisoformat(exp_str)
    dte = (exp - quote_date).days
    if dte < 7 or dte > 550:
        continue
    if dte > 92 and not is_third_friday(exp):
        continue
    expiries.append(exp_str)

print(f"{len(expiries)} expirations retenues :")
print(expiries)

15 expirations retenues :
['2026-07-10', '2026-07-17', '2026-07-24', '2026-07-31', '2026-08-07', '2026-08-21', '2026-09-18', '2026-10-16', '2026-11-20', '2026-12-18', '2027-01-15', '2027-02-19', '2027-03-19', '2027-09-17', '2027-12-17']


In [3]:
# Téléchargement leg par leg. On conserve bid/ask/last, volume, open interest et
# l'implied vol indicative de Yahoo (jamais utilisée pour calibrer, juste pour
# recouper). Schéma du CSV : voir JR_PRICER/market_data/loaders/option_chain.py.
COLUMNS = {"strike": "strike", "bid": "bid", "ask": "ask", "lastPrice": "last",
           "volume": "volume", "openInterest": "open_interest",
           "impliedVolatility": "iv_yf"}

frames = []
for exp_str in expiries:
    chain = tk.option_chain(exp_str)
    for leg, option_type in ((chain.calls, "C"), (chain.puts, "P")):
        part = leg[list(COLUMNS)].rename(columns=COLUMNS).copy()
        part.insert(0, "option_type", option_type)
        part.insert(0, "expiry", exp_str)
        frames.append(part)

chain_df = pd.concat(frames, ignore_index=True)
chain_df.insert(0, "spot", spot)
chain_df.insert(0, "quote_date", quote_date.isoformat())
print(f"{len(chain_df)} quotes téléchargées "
      f"({(chain_df.option_type == 'C').sum()} calls, {(chain_df.option_type == 'P').sum()} puts)")
chain_df.head()

1898 quotes téléchargées (1054 calls, 844 puts)


,quote_date,spot,expiry,option_type,strike,bid,ask,last,volume,open_interest,iv_yf
0,2026-07-02,305.700104,2026-07-10,C,130.0,172.65,175.80,160.40,NaN,1,2.105473
1,2026-07-02,305.700104,2026-07-10,C,155.0,147.65,150.95,125.60,2.0,1,1.896485
2,2026-07-02,305.700104,2026-07-10,C,185.0,117.65,120.90,97.58,1.0,1,1.400394
3,2026-07-02,305.700104,2026-07-10,C,195.0,107.75,111.15,94.42,1.0,2,1.424808
4,2026-07-02,305.700104,2026-07-10,C,200.0,102.70,105.75,76.24,1.0,2,1.019536


In [4]:
out = Path("data") / f"aapl_option_chain_{quote_date.isoformat()}.csv"
out.parent.mkdir(exist_ok=True)
chain_df.to_csv(out, index=False)
print(f"écrit : {out} ({out.stat().st_size / 1024:.0f} Ko)")

écrit : data/aapl_option_chain_2026-07-02.csv (169 Ko)


Le CSV est committé avec le repo : n'importe qui peut exécuter EQUITY_03 sans réseau
et retrouver exactement les mêmes chiffres.